# 04 — Synthetic ground-truth Yamada recovery

This notebook validates the full synthetic pipeline

$$
G_{\rm true}\rightarrow F(G_{\rm true})\rightarrow V_{200}
\rightarrow \widehat G\rightarrow \Upsilon(\widehat G;A)
$$

at the single fixed voxel resolution $N=200$.

Only connected, bridgeless, exactly trivalent ground-truth graphs are used.
After voxel recovery, Yamada is evaluated **only when every recovered vertex has
degree $\le 3$**. A recovered degree $>3$ is printed as `FAIL-DEG` and is treated
as a structural extraction failure. No plots are produced; all results are printed.


In [ ]:
from pathlib import Path
import json, hashlib, sys
import networkx as nx
import numpy as np
import sympy as sp
from skimage.morphology import ball, dilation, skeletonize

ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/"pyproject.toml").exists():
    ROOT=ROOT.parent
SRC=ROOT/"src"
if str(SRC) not in sys.path: sys.path.insert(0,str(SRC))

from knotted_graph.core import simplify_edges
from knotted_graph.extraction import skeleton_image_to_graph
from knotted_graph.projection import compute_yamada_polynomial

A=sp.Symbol("A")
BOUND=1.35
N=200
RADII=[1,2,3]
TRANSFORMS=["identity","rotate","affine"]
MAX_DEGREE=3
CHECKPOINT=ROOT/"User_guide"/"benchmarks"/"synthetic_ground_truth_results_n200_v5.jsonl"
RESUME=True
print("N =",N,"radii =",RADII,"transforms =",TRANSFORMS)


In [ ]:
def normalize(X,scale=.72):
    X=np.asarray(X,float); X-=X.mean(0)
    return X*(scale/np.max(np.linalg.norm(X,axis=1)))

def embedded_graph(G,pos):
    H=nx.MultiGraph()
    for n,p in pos.items(): H.add_node(n,pos=np.asarray(p,float))
    for u,v in G.edges(): H.add_edge(u,v,pts=np.linspace(pos[u],pos[v],80))
    return H

def theta_graph(bowed=False):
    t=np.linspace(0,1,500); x=-.72+1.44*t
    z=.16*np.sin(2*np.pi*t) if bowed else 0*t
    curves=[np.c_[x,-.58*np.sin(np.pi*t),z],
            np.c_[x,0*t,-.10*np.sin(np.pi*t) if bowed else 0*t],
            np.c_[x,.58*np.sin(np.pi*t),-z]]
    H=nx.MultiGraph(); H.add_node("u",pos=np.array([-.72,0,0.])); H.add_node("v",pos=np.array([.72,0,0.]))
    for P in curves:
        P[0]=[-.72,0,0]; P[-1]=[.72,0,0]; H.add_edge("u","v",pts=P)
    return H

def make_case(name,G,planar=True,seed=0):
    G=nx.Graph(G)
    assert nx.is_connected(G) and not list(nx.bridges(G)) and all(d==3 for _,d in G.degree())
    if planar:
        ok,_=nx.check_planarity(G); assert ok
        p=nx.planar_layout(G); X=normalize([[p[n][0],p[n][1],0] for n in G])
    else:
        p=nx.spring_layout(G,dim=3,seed=seed,iterations=1000); X=normalize([p[n] for n in G])
    pos={n:X[i] for i,n in enumerate(G)}
    return embedded_graph(G,pos)

CASES={
    "theta3_planar":theta_graph(False),
    "theta3_bowed":theta_graph(True),
    "K4":make_case("K4",nx.complete_graph(4),True),
    "triangular_prism":make_case("triangular_prism",nx.circular_ladder_graph(3),True),
    "cube":make_case("cube",nx.cubical_graph(),True),
    "pentagonal_prism":make_case("pentagonal_prism",nx.circular_ladder_graph(5),True),
    "dodecahedral":make_case("dodecahedral",nx.dodecahedral_graph(),True),
    "K3_3":make_case("K3_3",nx.complete_bipartite_graph(3,3),False,17),
    "petersen":make_case("petersen",nx.petersen_graph(),False,18),
    "heawood":make_case("heawood",nx.heawood_graph(),False,19),
}
for name,G in CASES.items():
    ds=dict(G.degree())
    assert ds and max(ds.values())==3 and min(ds.values())==3
    print(f"{name:20s} V={G.number_of_nodes():2d} E={G.number_of_edges():2d} max_degree=3")


In [ ]:
def Rxyz(a,b,c):
    a,b,c=np.deg2rad([a,b,c])
    Rx=np.array([[1,0,0],[0,np.cos(a),-np.sin(a)],[0,np.sin(a),np.cos(a)]])
    Ry=np.array([[np.cos(b),0,np.sin(b)],[0,1,0],[-np.sin(b),0,np.cos(b)]])
    Rz=np.array([[np.cos(c),-np.sin(c),0],[np.sin(c),np.cos(c),0],[0,0,1]])
    return Rz@Ry@Rx

def deform(G,name):
    if name=="identity": M,b=np.eye(3),np.zeros(3)
    elif name=="rotate": M,b=Rxyz(21,34,13),np.array([.04,-.03,.02])
    else:
        M=Rxyz(17,-23,31)@np.diag([1.08,.91,1.03])@np.array([[1,.13,0],[0,1,.09],[.05,0,1]])
        b=np.array([-.03,.04,-.02])
    assert np.linalg.det(M)>0
    H=nx.MultiGraph()
    for n,d in G.nodes(data=True): H.add_node(n,pos=np.asarray(d["pos"])@M.T+b)
    for u,v,k,d in G.edges(keys=True,data=True): H.add_edge(u,v,pts=np.asarray(d["pts"])@M.T+b)
    return H

def voxelize(G,r):
    V=np.zeros((N,N,N),bool); dx=2*BOUND/(N-1)
    for *_,d in G.edges(keys=True,data=True):
        P=np.asarray(d["pts"]); Q=[]
        for p,q in zip(P[:-1],P[1:]):
            m=max(2,int(np.ceil(np.linalg.norm(q-p)/(dx/3)))+1)
            Q.append(np.linspace(p,q,m,endpoint=False))
        Q=np.vstack(Q+[P[-1:]])
        I=np.rint((Q+BOUND)/(2*BOUND)*(N-1)).astype(int)
        I=np.clip(I,0,N-1); V[I[:,0],I[:,1],I[:,2]]=1
    return dilation(V,footprint=ball(r))

def recover(V):
    raw=skeleton_image_to_graph(skeletonize(V,method="lee"))
    H=nx.MultiGraph(raw); dx=2*BOUND/(N-1); o=np.array([-BOUND]*3)
    for _,d in H.nodes(data=True): d["pos"]=o+dx*np.asarray(d["pos"],float)
    for *_,d in H.edges(keys=True,data=True): d["pts"]=o+dx*np.asarray(d["pts"],float)
    return simplify_edges(H)

def yamada(G):
    bad={n:d for n,d in G.degree() if d>MAX_DEGREE}
    if bad: raise ValueError(f"Yamada refused: recovered degree > 3: {bad}")
    out=compute_yamada_polynomial(G,A,num_rotation_samples=16,crossing_warning_threshold=None,
                                  normalize=True,n_jobs=1,method="recursive",return_result=True)
    return sp.expand(out.polynomial),out.projection

def same(a,b):
    return sp.simplify(sp.together(sp.expand(a-b)))==0


In [ ]:
TARGETS={}
print("GROUND-TRUTH CHECK")
for name,G in CASES.items():
    assert all(d==3 for _,d in G.degree())
    TARGETS[name],p=yamada(G)
    print(f"TARGET {name:20s} crossings={p.num_crossings:2d} Yamada={TARGETS[name]}")
    for t in TRANSFORMS:
        H=deform(G,t)
        assert max(dict(H.degree()).values())==3
        poly,_=yamada(H)
        if not same(poly,TARGETS[name]):
            raise AssertionError(f"{name}/{t}: Yamada changed before voxelization")
print("PASS: all ground-truth/deformed graphs remain trivalent and Yamada-equivalent.")


In [ ]:
signature=hashlib.sha256(json.dumps({"N":N,"r":RADII,"t":TRANSFORMS,"cases":list(CASES)},sort_keys=True).encode()).hexdigest()[:20]
done={}
if RESUME and CHECKPOINT.exists():
    for line in CHECKPOINT.read_text().splitlines():
        if line.strip():
            x=json.loads(line)
            if x.get("signature")==signature:
                done[(x["case"],x["transform"],x["radius_vox"])]=x

records=[]; CHECKPOINT.parent.mkdir(parents=True,exist_ok=True)
for name,G0 in CASES.items():
    for t in TRANSFORMS:
        G=deform(G0,t)
        for r in RADII:
            k=(name,t,r)
            if k in done:
                row=done[k]; records.append(row)
                print("CACHED",name,t,"r=",r,"success=",row["success"],"reason=",row["reason"])
                continue
            row={"signature":signature,"case":name,"transform":t,"resolution":N,"radius_vox":r,
                 "success":False,"final_V":None,"final_E":None,"max_degree":None,
                 "yamada_evaluated":False,"reason":None,"recovered_yamada":None,"error":None}
            try:
                H=recover(voxelize(G,r))
                row["final_V"]=H.number_of_nodes(); row["final_E"]=H.number_of_edges()
                row["max_degree"]=max((d for _,d in H.degree()),default=0)
                if row["max_degree"]>3:
                    row["reason"]="recovered_degree_gt_3"
                else:
                    poly,p=yamada(H); row["yamada_evaluated"]=True; row["recovered_yamada"]=str(poly)
                    row["success"]=bool(same(poly,TARGETS[name]))
                    row["reason"]=None if row["success"] else "yamada_mismatch"
                    row["crossings"]=p.num_crossings
            except Exception as exc:
                row["reason"]="execution_error"; row["error"]=f"{type(exc).__name__}: {exc}"
            records.append(row)
            with CHECKPOINT.open("a") as f: f.write(json.dumps(row)+"\n")
            mark="PASS" if row["success"] else ("FAIL-DEG" if row["reason"]=="recovered_degree_gt_3" else "FAIL")
            print(f"{mark:8s} {name:20s} {t:8s} N={N} r={r} V/E={row['final_V']}/{row['final_E']} "
                  f"maxdeg={row['max_degree']} Yamada={'yes' if row['yamada_evaluated'] else 'no'} reason={row['reason']}")

print("\nOVERALL")
passed=sum(x["success"] for x in records)
degfail=sum(x["reason"]=="recovered_degree_gt_3" for x in records)
mismatch=sum(x["reason"]=="yamada_mismatch" for x in records)
errors=sum(x["reason"]=="execution_error" for x in records)
print(f"resolution: N={N}")
print(f"exact recoveries: {passed}/{len(records)} = {100*passed/len(records):.2f}%")
print(f"degree>3 structural failures: {degfail}")
print(f"Yamada mismatches with degree<=3: {mismatch}")
print(f"execution errors: {errors}")

for label,key,vals in [
    ("PER GRAPH","case",list(CASES)),
    ("PER TRANSFORM","transform",TRANSFORMS),
    ("PER RADIUS","radius_vox",RADII),
]:
    print("\n"+label)
    for v in vals:
        g=[x for x in records if x[key]==v]
        p=sum(x["success"] for x in g)
        d=sum(x["reason"]=="recovered_degree_gt_3" for x in g)
        m=sum(x["reason"]=="yamada_mismatch" for x in g)
        e=sum(x["reason"]=="execution_error" for x in g)
        print(f"{str(v):20s} pass={p:2d}/{len(g):2d} ({100*p/len(g):6.2f}%) degree>3={d:2d} mismatch={m:2d} errors={e:2d}")


### Interpretation

- `PASS`: recovered graph has degree at most 3 and exactly matches the ground-truth Yamada polynomial.
- `FAIL-DEG`: voxel recovery created a vertex of degree greater than 3; Yamada was **not** called.
- `FAIL` with `yamada_mismatch`: recovered graph stayed in the supported degree range but its Yamada polynomial differs.
- The only voxel resolution used is $N=200$.
- No plots are generated.
